In [ ]:

# STEP 106: Load Incremental Bronze Tables

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

# Read production incremental Bronze tables
customers_bronze_df = spark.read.table("bronze_customers_incremental")
products_bronze_df = spark.read.table("bronze_products_incremental")
sales_bronze_df = spark.read.table("bronze_sales_incremental")

# Verify source record counts
print("=== INCREMENTAL BRONZE SOURCE COUNTS ===")
print("Customers:", customers_bronze_df.count())
print("Products:", products_bronze_df.count())
print("Sales:", sales_bronze_df.count())

# Verify schemas
print("\n=== CUSTOMERS SCHEMA ===")
customers_bronze_df.printSchema()

print("\n=== PRODUCTS SCHEMA ===")
products_bronze_df.printSchema()

print("\n=== SALES SCHEMA ===")
sales_bronze_df.printSchema()


In [ ]:

# STEP 107: Customer Data Quality Validation

# Identify invalid customer records
invalid_customers_df = customers_bronze_df.filter(
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "") |
    F.col("customer_name").isNull() |
    (F.trim(F.col("customer_name")) == "") |
    F.col("signup_date").isNull() |
    F.col("is_deleted").isNull() |
    (
        (F.col("is_deleted") == True) &
        F.col("deleted_at").isNull()
    ) |
    (
        (F.col("is_deleted") == False) &
        F.col("deleted_at").isNotNull()
    )
)

# Identify duplicate customer IDs
duplicate_customers_df = (
    customers_bronze_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

# Build validated Silver Customer DataFrame
silver_customers_df = customers_bronze_df.subtract(
    invalid_customers_df
)

# Check validation results
print("=== CUSTOMER DATA QUALITY RESULTS ===")
print("Bronze Customers:", customers_bronze_df.count())
print("Invalid Customers:", invalid_customers_df.count())
print("Duplicate Customer IDs:", duplicate_customers_df.count())
print("Validated Customers:", silver_customers_df.count())

# Stop processing if duplicate IDs exist
if duplicate_customers_df.count() > 0:
    raise ValueError("Duplicate customer IDs detected in Bronze!")

print("\n=== VALIDATED CUSTOMERS ===")
silver_customers_df.orderBy("customer_id").show(truncate=False)


In [ ]:

# STEP 108: Product Data Quality Validation

# Convert Product unit_price to Decimal
products_typed_df = products_bronze_df.withColumn(
    "unit_price",
    F.col("unit_price").cast("decimal(12,2)")
)

# Identify invalid Products
invalid_products_df = products_typed_df.filter(
    F.col("product_id").isNull() |
    (F.trim(F.col("product_id")) == "") |
    F.col("product_name").isNull() |
    (F.trim(F.col("product_name")) == "") |
    F.col("category").isNull() |
    (F.trim(F.col("category")) == "") |
    F.col("unit_price").isNull() |
    (F.col("unit_price") <= 0) |
    F.col("is_deleted").isNull() |
    (
        (F.col("is_deleted") == True) &
        F.col("deleted_at").isNull()
    ) |
    (
        (F.col("is_deleted") == False) &
        F.col("deleted_at").isNotNull()
    )
)

# Check duplicate Product IDs
duplicate_products_df = (
    products_typed_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

# Keep valid Products
silver_products_df = products_typed_df.filter(
    ~(
        F.col("product_id").isNull() |
        (F.trim(F.col("product_id")) == "") |
        F.col("product_name").isNull() |
        (F.trim(F.col("product_name")) == "") |
        F.col("category").isNull() |
        (F.trim(F.col("category")) == "") |
        F.col("unit_price").isNull() |
        (F.col("unit_price") <= 0) |
        F.col("is_deleted").isNull() |
        (
            (F.col("is_deleted") == True) &
            F.col("deleted_at").isNull()
        ) |
        (
            (F.col("is_deleted") == False) &
            F.col("deleted_at").isNotNull()
        )
    )
)

# Stop if duplicate IDs exist
if duplicate_products_df.count() > 0:
    raise ValueError("Duplicate Product IDs detected in Bronze!")

# Print results
print("=== PRODUCT DATA QUALITY RESULTS ===")
print("Bronze Products:", products_bronze_df.count())
print("Invalid Products:", invalid_products_df.count())
print("Duplicate Product IDs:", duplicate_products_df.count())
print("Validated Products:", silver_products_df.count())

print("\n=== VALIDATED PRODUCTS ===")
silver_products_df.orderBy("product_id").show(truncate=False)

print("\n=== SILVER PRODUCTS SCHEMA ===")
silver_products_df.printSchema()


In [ ]:

# STEP 109: Sales Data Quality Validation

# Safely convert Sales data types
sales_typed_df = (
    sales_bronze_df
    .withColumn(
        "sale_date",
        F.expr("try_cast(sale_date AS DATE)")
    )
    .withColumn(
        "quantity",
        F.expr("try_cast(quantity AS INT)")
    )
    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )
)

# Identify invalid Sales records
invalid_sales_quality_df = sales_typed_df.filter(
    F.col("sale_id").isNull() |
    (F.trim(F.col("sale_id")) == "") |
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "") |
    F.col("product_id").isNull() |
    (F.trim(F.col("product_id")) == "") |
    F.col("sale_date").isNull() |
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0) |
    F.col("unit_price").isNull() |
    (F.col("unit_price") <= 0)
)

# Identify duplicate Sale IDs
duplicate_sales_df = (
    sales_typed_df
    .groupBy("sale_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check data quality counts
print("=== SALES DATA QUALITY RESULTS ===")
print("Bronze Sales:", sales_bronze_df.count())
print("Invalid Sales:", invalid_sales_quality_df.count())
print("Duplicate Sale IDs:", duplicate_sales_df.count())

# Stop if duplicate IDs exist
if duplicate_sales_df.count() > 0:
    raise ValueError("Duplicate Sale IDs detected in Bronze!")

# Keep Sales passing data quality checks
silver_sales_quality_df = sales_typed_df.join(
    invalid_sales_quality_df.select("sale_id"),
    on="sale_id",
    how="left_anti"
)

print("Validated Sales:", silver_sales_quality_df.count())

print("\n=== SILVER SALES SCHEMA ===")
silver_sales_quality_df.printSchema()

print("\n=== SAMPLE VALIDATED SALES ===")
silver_sales_quality_df.orderBy("sale_id").show(10, truncate=False)


In [ ]:

# STEP 109A: Investigate Invalid Sales Records

invalid_sales_with_reason_df = (
    invalid_sales_quality_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            " | ",
            F.when(
                F.col("sale_id").isNull() |
                (F.trim(F.col("sale_id")) == ""),
                "INVALID_SALE_ID"
            ),
            F.when(
                F.col("customer_id").isNull() |
                (F.trim(F.col("customer_id")) == ""),
                "INVALID_CUSTOMER_ID"
            ),
            F.when(
                F.col("product_id").isNull() |
                (F.trim(F.col("product_id")) == ""),
                "INVALID_PRODUCT_ID"
            ),
            F.when(
                F.col("sale_date").isNull(),
                "INVALID_SALE_DATE"
            ),
            F.when(
                F.col("quantity").isNull() |
                (F.col("quantity") <= 0),
                "INVALID_QUANTITY"
            ),
            F.when(
                F.col("unit_price").isNull() |
                (F.col("unit_price") <= 0),
                "INVALID_UNIT_PRICE"
            )
        )
    )
)

print("=== INVALID SALES WITH REJECTION REASONS ===")

invalid_sales_with_reason_df.select(
    "sale_id",
    "customer_id",
    "product_id",
    "sale_date",
    "quantity",
    "unit_price",
    "rejection_reason"
).orderBy("sale_id").show(truncate=False)


In [ ]:

# STEP 110: Silver Sales Referential Integrity Validation

# Select active, validated Customers and Products
active_customers_df = silver_customers_df.filter(
    F.col("is_deleted") == False
).select("customer_id")

active_products_df = silver_products_df.filter(
    F.col("is_deleted") == False
).select("product_id")

# Keep Sales referencing active Customers
customer_valid_sales_df = silver_sales_quality_df.join(
    active_customers_df,
    on="customer_id",
    how="left_semi"
)

# Keep Sales referencing active Products
silver_sales_df = customer_valid_sales_df.join(
    active_products_df,
    on="product_id",
    how="left_semi"
)

# Identify Sales rejected by referential validation
referential_rejected_sales_df = silver_sales_quality_df.join(
    silver_sales_df.select("sale_id"),
    on="sale_id",
    how="left_anti"
)

# Reconciliation counts
quality_valid_count = silver_sales_quality_df.count()
referential_rejected_count = referential_rejected_sales_df.count()
silver_sales_count = silver_sales_df.count()

print("=== SILVER REFERENTIAL INTEGRITY RESULTS ===")
print("Quality-validated Sales:", quality_valid_count)
print("Referentially rejected Sales:", referential_rejected_count)
print("Final valid Silver Sales:", silver_sales_count)

if quality_valid_count != silver_sales_count + referential_rejected_count:
    raise ValueError("Silver Sales reconciliation failed!")

print("\n=== FINAL SILVER SALES SAMPLE ===")
silver_sales_df.orderBy("sale_id").show(10, truncate=False)


In [ ]:

# STEP 111: Incremental Silver Customers Delta MERGE

silver_customers_table = "dbo.silver_customers_incremental"

# Create table on first run
if not spark.catalog.tableExists(silver_customers_table):

    silver_customers_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(silver_customers_table)

    print("Silver Customers table created successfully!")

else:

    # Merge validated Customers into existing Silver table
    delta_customers = DeltaTable.forName(
        spark,
        silver_customers_table
    )

    (
        delta_customers.alias("target")
        .merge(
            silver_customers_df.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Silver Customers MERGE completed successfully!")

# Verify saved table
saved_silver_customers_df = spark.read.table(
    silver_customers_table
)

print("\n=== SILVER CUSTOMERS VALIDATION ===")
print("Source Customers:", silver_customers_df.count())
print("Saved Customers:", saved_silver_customers_df.count())

saved_silver_customers_df.orderBy(
    "customer_id"
).show(truncate=False)


In [ ]:

# STEP 112: Incremental Silver Products Delta MERGE

silver_products_table = "dbo.silver_products_incremental"

# Create table on first run
if not spark.catalog.tableExists(silver_products_table):

    silver_products_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(silver_products_table)

    print("Silver Products table created successfully!")

else:

    # Merge validated Products into existing Silver table
    delta_products = DeltaTable.forName(
        spark,
        silver_products_table
    )

    (
        delta_products.alias("target")
        .merge(
            silver_products_df.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Silver Products MERGE completed successfully!")

# Verify saved table
saved_silver_products_df = spark.read.table(
    silver_products_table
)

print("\n=== SILVER PRODUCTS VALIDATION ===")
print("Source Products:", silver_products_df.count())
print("Saved Products:", saved_silver_products_df.count())

saved_silver_products_df.orderBy(
    "product_id"
).show(truncate=False)

print("\n=== SAVED PRODUCTS SCHEMA ===")
saved_silver_products_df.printSchema()


In [ ]:

# STEP 113: Incremental Silver Sales Delta MERGE

silver_sales_table = "dbo.silver_sales_incremental"

# Validate source uniqueness before writing
duplicate_count = (
    silver_sales_df
    .groupBy("sale_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_count > 0:
    raise ValueError("Duplicate Sale IDs found in Silver source!")

# Create Silver Sales table on first run
if not spark.catalog.tableExists(silver_sales_table):

    silver_sales_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(silver_sales_table)

    print("Silver Sales table created successfully!")

else:

    # Synchronize Sales with validated active source
    delta_sales = DeltaTable.forName(
        spark,
        silver_sales_table
    )

    (
        delta_sales.alias("target")
        .merge(
            silver_sales_df.alias("source"),
            "target.sale_id = source.sale_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Silver Sales MERGE completed successfully!")

# Verify saved records
saved_silver_sales_df = spark.read.table(
    silver_sales_table
)

source_count = silver_sales_df.count()
saved_count = saved_silver_sales_df.count()

print("\n=== SILVER SALES VALIDATION ===")
print("Validated Sales:", source_count)
print("Saved Silver Sales:", saved_count)

if source_count != saved_count:
    raise ValueError("Silver Sales record count mismatch!")

print("\n=== SAVED SILVER SALES SCHEMA ===")
saved_silver_sales_df.printSchema()

print("\n=== SAMPLE SILVER SALES ===")
saved_silver_sales_df.orderBy(
    "sale_id"
).show(10, truncate=False)


In [ ]:

# STEP 114: Incremental Silver Rejected Sales

rejected_sales_table = "dbo.rejected_sales_incremental"

# Combine quality and referential rejections
referential_rejected_with_reason_df = (
    referential_rejected_sales_df
    .withColumn(
        "rejection_reason",
        F.lit("INVALID_CUSTOMER_OR_PRODUCT_REFERENCE")
    )
)

rejected_sales_source_df = (
    invalid_sales_with_reason_df
    .unionByName(referential_rejected_with_reason_df)
)

# Validate unique Sale IDs
duplicate_rejected_ids = (
    rejected_sales_source_df
    .groupBy("sale_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_rejected_ids > 0:
    raise ValueError("Duplicate Sale IDs found in rejected Sales!")

# Create table on first run
if not spark.catalog.tableExists(rejected_sales_table):

    rejected_sales_source_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(rejected_sales_table)

    print("Rejected Sales table created successfully!")

else:

    delta_rejected = DeltaTable.forName(
        spark,
        rejected_sales_table
    )

    (
        delta_rejected.alias("target")
        .merge(
            rejected_sales_source_df.alias("source"),
            "target.sale_id = source.sale_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Rejected Sales MERGE completed successfully!")

# Verify saved rejected Sales
saved_rejected_sales_df = spark.read.table(
    rejected_sales_table
)

print("\n=== SILVER REJECTED SALES VALIDATION ===")
print("Source Rejected Sales:", rejected_sales_source_df.count())
print("Saved Rejected Sales:", saved_rejected_sales_df.count())

saved_rejected_sales_df.select(
    "sale_id",
    "rejection_reason"
).orderBy("sale_id").show(truncate=False)


In [ ]:

# STEP 115: Final Incremental Silver Validation

from pyspark.sql.types import DateType, IntegerType, DecimalType

# Read persisted Silver Delta tables
sc = spark.read.table("dbo.silver_customers_incremental")
sp = spark.read.table("dbo.silver_products_incremental")
ss = spark.read.table("dbo.silver_sales_incremental")
sr = spark.read.table("dbo.rejected_sales_incremental")

# 1. Validate unique, non-null business keys
for table_name, df, key in [
    ("Silver Customers", sc, "customer_id"),
    ("Silver Products", sp, "product_id"),
    ("Silver Sales", ss, "sale_id"),
    ("Rejected Sales", sr, "sale_id")
]:
    duplicates = df.groupBy(key).count().filter(
        F.col("count") > 1
    ).count()

    missing_keys = df.filter(
        F.col(key).isNull() |
        (F.trim(F.col(key)) == "")
    ).count()

    if duplicates > 0 or missing_keys > 0:
        raise ValueError(
            f"{table_name}: duplicate or missing business keys!"
        )

# 2. Check overlap between valid and rejected Sales
overlap_count = ss.select("sale_id").join(
    sr.select("sale_id"),
    on="sale_id",
    how="inner"
).count()

if overlap_count > 0:
    raise ValueError("Sales overlap between valid and rejected Silver!")

# 3. Reconcile Bronze Sales with Silver outcomes
bronze_count = sales_bronze_df.count()
silver_count = ss.count()
rejected_count = sr.count()

if bronze_count != silver_count + rejected_count:
    raise ValueError(
        "Bronze-to-Silver Sales reconciliation failed!"
    )

# 4. Verify every Bronze Sale ID appears in Silver exactly once
bronze_ids = sales_bronze_df.select("sale_id")
outcome_ids = ss.select("sale_id").unionByName(
    sr.select("sale_id")
)

missing_outcomes = bronze_ids.join(
    outcome_ids,
    on="sale_id",
    how="left_anti"
).count()

unexpected_outcomes = outcome_ids.join(
    bronze_ids,
    on="sale_id",
    how="left_anti"
).count()

if missing_outcomes > 0 or unexpected_outcomes > 0:
    raise ValueError("Silver Sales IDs do not match Bronze!")

# 5. Validate active Customer and Product references
active_customer_ids = sc.filter(
    F.col("is_deleted") == False
).select("customer_id")

active_product_ids = sp.filter(
    F.col("is_deleted") == False
).select("product_id")

invalid_customer_refs = ss.join(
    active_customer_ids,
    on="customer_id",
    how="left_anti"
).count()

invalid_product_refs = ss.join(
    active_product_ids,
    on="product_id",
    how="left_anti"
).count()

if invalid_customer_refs > 0 or invalid_product_refs > 0:
    raise ValueError("Invalid Silver Sales references detected!")

# 6. Verify Silver Sales data types
assert isinstance(ss.schema["sale_date"].dataType, DateType)
assert isinstance(ss.schema["quantity"].dataType, IntegerType)
assert ss.schema["unit_price"].dataType == DecimalType(12, 2)

print("=== FINAL INCREMENTAL SILVER VALIDATION ===")
print("Silver Customers:", sc.count())
print("Silver Products:", sp.count())
print("Bronze Sales:", bronze_count)
print("Valid Silver Sales:", silver_count)
print("Rejected Silver Sales:", rejected_count)
print("Duplicate business keys: 0")
print("Valid/Rejected overlap:", overlap_count)
print("Missing Sales outcomes:", missing_outcomes)
print("Unexpected Sales outcomes:", unexpected_outcomes)
print("Invalid Customer references:", invalid_customer_refs)
print("Invalid Product references:", invalid_product_refs)

print("\nFINAL INCREMENTAL SILVER VALIDATION PASSED")


In [ ]:

# STEP 116D: Verify Silver Customers Table

table_name = "dbo.silver_customers_incremental"

print("Table exists:", spark.catalog.tableExists(table_name))

history_df = spark.sql(
    "DESCRIBE HISTORY dbo.silver_customers_incremental"
)

history_df.select(
    "version",
    "operation",
    "timestamp"
).orderBy("version", ascending=False).show(5, truncate=False)


In [ ]:

# STEP 116E: Verify Silver Delta Table Histories

tables = [
    "dbo.silver_products_incremental",
    "dbo.silver_sales_incremental",
    "dbo.rejected_sales_incremental"
]

for table_name in tables:
    print(f"\n=== {table_name} ===")

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    history_df.select(
        "version",
        "operation",
        "timestamp"
    ).orderBy(
        "version",
        ascending=False
    ).show(5, truncate=False)

    print(
        "Current records:",
        spark.read.table(table_name).count()
    )
